# SentIQ — Brand Sentiment Intelligence Platform

**Problem Statement:**  
Companies struggle to monitor public sentiment about their brand in real time. This notebook builds an end-to-end AI pipeline that collects live news data, runs sentiment analysis, and produces actionable business insights.

**Demo:** MediaTek vs Snapdragon sentiment comparison using real news articles.

**Pipeline:**
1. Data collection via NewsAPI
2. Storage in Databricks Delta tables
3. SQL analysis
4. AI sentiment analysis (HuggingFace DistilBERT)
5. A/B testing with statistical significance
6. Visualisation and export for Power BI

In [0]:
%pip install newsapi-python transformers torch

## 1. Data Collection

In [0]:
from newsapi import NewsApiClient
import pandas as pd

# Store API key securely in production using Databricks secrets
API_KEY = 'YOUR_NEWSAPI_KEY_HERE'
newsapi = NewsApiClient(api_key=API_KEY)

brands = ['MediaTek', 'Snapdragon']
all_articles = []

for brand in brands:
    articles = newsapi.get_everything(
        q=brand,
        language='en',
        sort_by='publishedAt',
        page_size=100
    )
    for article in articles['articles']:
        all_articles.append({
            'brand': brand,
            'title': article['title'],
            'description': article['description'],
            'published_at': article['publishedAt'],
            'source': article['source']['name'],
            'url': article['url']
        })

df = pd.DataFrame(all_articles)
print(f"Total articles collected: {len(df)}")
print(df.head())

## 2. Storage — Databricks Delta Table

In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.format('delta').mode('overwrite').saveAsTable('sentiq_raw_news')
print(f"Saved {spark_df.count()} articles to sentiq_raw_news")

## 3. SQL Analysis

In [0]:
spark.sql("""
    SELECT 
        brand,
        COUNT(*) as total_articles,
        COUNT(DISTINCT source) as unique_sources,
        MIN(published_at) as earliest_article,
        MAX(published_at) as latest_article
    FROM sentiq_raw_news
    GROUP BY brand
    ORDER BY total_articles DESC
""").show()

## 4. Sentiment Analysis

In [0]:
from transformers import pipeline

sentiment_model = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

def get_sentiment(text):
    if not text or text == 'None':
        return 'NEUTRAL', 0.0
    try:
        result = sentiment_model(text[:512])[0]
        return result['label'], round(result['score'], 3)
    except:
        return 'NEUTRAL', 0.0

df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
df['sentiment'], df['confidence'] = zip(*df['text'].apply(get_sentiment))
df['sentiment_score'] = (df['sentiment'] == 'POSITIVE').astype(int)

print(df[['brand', 'sentiment', 'confidence']].value_counts().head(10))

## 5. KPI Summary

In [0]:
summary = df.groupby(['brand', 'sentiment']).size().reset_index(name='count')
summary['percentage'] = summary.groupby('brand')['count'].transform(
    lambda x: round(x / x.sum() * 100, 1)
)

print(summary.to_string(index=False))
print()
print(df.groupby(['brand', 'sentiment'])['confidence'].mean().round(3))

## 6. A/B Test — Statistical Significance

- **H₀:** No significant difference in sentiment between brands
- **Test:** Independent samples t-test
- **α = 0.05**

In [0]:
from scipy import stats

mediatek_scores = (df[df['brand'] == 'MediaTek']['sentiment'] == 'POSITIVE').astype(int).values
snapdragon_scores = (df[df['brand'] == 'Snapdragon']['sentiment'] == 'POSITIVE').astype(int).values

t_stat, p_value = stats.ttest_ind(mediatek_scores, snapdragon_scores)
alpha = 0.05

print(f"MediaTek positive rate:   {mediatek_scores.mean():.1%}")
print(f"Snapdragon positive rate: {snapdragon_scores.mean():.1%}")
print(f"T-statistic:              {t_stat:.3f}")
print(f"P-value:                  {p_value:.4f}")
print(f"Conclusion: {'Reject H0 — significant difference' if p_value < alpha else 'Fail to reject H0 — no significant difference at alpha=0.05'}")

## 7. Visualisation

In [0]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SentIQ — Brand Sentiment Analysis', fontsize=16, fontweight='bold')

brands_list = ['MediaTek', 'Snapdragon']
positive = [
    summary[(summary['brand']=='MediaTek') & (summary['sentiment']=='POSITIVE')]['percentage'].values[0],
    summary[(summary['brand']=='Snapdragon') & (summary['sentiment']=='POSITIVE')]['percentage'].values[0]
]
negative = [
    summary[(summary['brand']=='MediaTek') & (summary['sentiment']=='NEGATIVE')]['percentage'].values[0],
    summary[(summary['brand']=='Snapdragon') & (summary['sentiment']=='NEGATIVE')]['percentage'].values[0]
]
x = np.arange(len(brands_list))
width = 0.35

axes[0].bar(x - width/2, positive, width, label='Positive', color='#2ecc71')
axes[0].bar(x + width/2, negative, width, label='Negative', color='#e74c3c')
axes[0].set_title('Sentiment Distribution (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(brands_list)
axes[0].set_ylabel('Percentage (%)')
axes[0].legend()
axes[0].set_ylim(0, 100)
for i, (p, n) in enumerate(zip(positive, negative)):
    axes[0].text(i - width/2, p + 1, f'{p}%', ha='center', fontsize=10)
    axes[0].text(i + width/2, n + 1, f'{n}%', ha='center', fontsize=10)

axes[1].pie([99, 98], labels=brands_list, colors=['#3498db', '#e67e22'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Article Volume by Brand')

plt.tight_layout()
plt.show()

In [0]:
df['published_at'] = pd.to_datetime(df['published_at'])
df['date'] = df['published_at'].dt.date

trend = df.groupby(['date', 'brand'])['sentiment_score'].mean().reset_index()

plt.figure(figsize=(14, 5))
for brand, color, marker in [('MediaTek', '#3498db', 'o'), ('Snapdragon', '#e67e22', 's')]:
    t = trend[trend['brand'] == brand]
    plt.plot(t['date'], t['sentiment_score'], marker=marker,
             label=brand, color=color, linewidth=2)

plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Neutral (0.5)')
plt.title('SentIQ — Daily Sentiment Trend', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Positive Sentiment Rate')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Export for Power BI

In [0]:
spark.createDataFrame(df).write.format('delta').mode('overwrite').saveAsTable('sentiq_sentiment_results')
spark.createDataFrame(summary).write.format('delta').mode('overwrite').saveAsTable('sentiq_summary')

user = spark.sql('SELECT current_user()').collect()[0][0]
save_path = f'/Workspace/Users/{user}'

df.to_csv(f'{save_path}/sentiq_sentiment_results.csv', index=False)
summary.to_csv(f'{save_path}/sentiq_summary.csv', index=False)

trend_pct = df.groupby(['date', 'brand'])['sentiment_score'].mean().reset_index()
trend_pct['date'] = trend_pct['date'].astype(str)
trend_pct['sentiment_score'] = (trend_pct['sentiment_score'].astype(float) * 100).round(1)
trend_pct.to_csv(f'{save_path}/sentiq_trend_pct.csv', index=False)

print(f"Exported to {save_path}")